In [6]:
import requests
import pandas as pd
import time
import os
from typing import Optional
from io import StringIO
from tqdm.auto import tqdm

d:\ALL CODES\REINCEFORCEMENT LEARNING\Lux Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
data=pd.read_excel("DATASETS\PGCB_date_power_demand.xlsx").head(10)
data.columns

Index(['datetime', 'generation_mw', 'demand_mw', 'load_shedding', 'gas',
       'liquid_fuel', 'coal', 'hydro', 'solar', 'wind', 'india_bheramara_hvdc',
       'india_tripura', 'india_adani', 'nepal', 'remarks'],
      dtype='object')

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [6]:
import lxml

In [1]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from tqdm import tqdm

In [9]:
def _normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
    )


def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    expected = [
        "datetime",
        "generation_mw",
        "demand_mw",
        "load_shedding",
        "gas",
        "liquid_fuel",
        "coal",
        "hydro",
        "solar",
        "wind",
        "india_bheramara_hvdc",
        "india_tripura",
        "india_adani",
        "nepal",
        "remarks",
    ]
    aliases = {
        "date_time": "datetime",
        "date": "datetime",
        "time": "datetime",
        "generation": "generation_mw",
        "demand": "demand_mw",
        "loadshedding": "load_shedding",
        "load_shedding_(mw)": "load_shedding",
        "liquidfuel": "liquid_fuel",
        "bheramara_hvdc": "india_bheramara_hvdc",
        "tripura": "india_tripura",
        "adani": "india_adani",
    }

    normalized = [_normalize_column_name(col) for col in df.columns]
    rename_map = {}
    for col, norm in zip(df.columns, normalized):
        target = aliases.get(norm, norm)
        if target in expected:
            rename_map[col] = target

    df = df.rename(columns=rename_map)
    return df


def _strip_repeated_header_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    normalized_cols = [_normalize_column_name(c) for c in df.columns]
    header_mask = df.apply(
        lambda row: [_normalize_column_name(v) for v in row.values] == normalized_cols,
        axis=1,
    )
    return df.loc[~header_mask]


def _extract_table_from_html(page_source: str) -> pd.DataFrame:
    tables = pd.read_html(StringIO(page_source))
    if not tables:
        return pd.DataFrame()

    standardized = []
    for table in tables:
        if table.empty:
            continue
        table = _strip_repeated_header_rows(table)
        table = _standardize_columns(table)
        standardized.append(table)

    if not standardized:
        return pd.DataFrame()

    expected_cols = {
        "datetime",
        "generation_mw",
        "demand_mw",
        "load_shedding",
        "gas",
        "liquid_fuel",
        "coal",
        "hydro",
        "solar",
        "wind",
        "india_bheramara_hvdc",
        "india_tripura",
        "india_adani",
        "nepal",
        "remarks",
    }

    def score(df: pd.DataFrame) -> int:
        return len(set(df.columns) & expected_cols)

    best = max(standardized, key=lambda df: (score(df), len(df)))
    return best


def scrape_powergrid_data_selenium(
    start_page: int = 1,
    end_page: int = 2,
    output_file: str = "powergrid_data.csv",
    save_html_dir: Optional[str] = "html_pages",
) -> pd.DataFrame:
    """
    Scrapes power generation data from erp.powergrid.gov.bd using Selenium.
    Downloads page HTML and extracts the main table.
    """
    base_url = "https://erp.powergrid.gov.bd/w/generations/view_generations"
    all_data = []

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")

    if save_html_dir:
        os.makedirs(save_html_dir, exist_ok=True)

    print(f"Starting scrape from page {start_page} to {end_page}...")

    driver = webdriver.Chrome(options=options)
    try:
        page_iter = tqdm(range(start_page, end_page + 1), desc="Pages", leave=True)
        for page in page_iter:
            url = f"{base_url}?page={page}"

            driver.get(url)
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )

            page_source = driver.page_source

            if save_html_dir:
                html_path = os.path.join(save_html_dir, f"page_{page}.html")
                with open(html_path, "w", encoding="utf-8") as html_file:
                    html_file.write(page_source)

            df = _extract_table_from_html(page_source)
            if df.empty:
                print("[Empty Data] - Skipping.")
                continue

            all_data.append(df)
            page_iter.set_postfix(rows=len(df))

            time.sleep(1)
    except Exception as e:
        print(f"[Error]: {e}")
    finally:
        driver.quit()

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"\nScraping complete. Saved {len(final_df)} rows to {output_file}")
        print("Columns detected:")
        print(final_df.columns)
        return final_df

    print("\nNo data extracted.")
    return pd.DataFrame()


if __name__ == "__main__":
    data = scrape_powergrid_data_selenium(start_page=1, end_page=120)

Starting scrape from page 1 to 120...


Pages: 100%|██████████| 120/120 [03:01<00:00,  1.51s/it, rows=51]



Scraping complete. Saved 6120 rows to powergrid_data.csv
Columns detected:
MultiIndex([(          'Date',           'Date'),
            (          'Time',           'Time'),
            ('Generation(MW)', 'Generation(MW)'),
            (    'Demand(MW)',     'Demand(MW)'),
            (      'Loadshed',       'Loadshed'),
            (           'Gas',            'Gas'),
            (   'Liquid Fuel',    'Liquid Fuel'),
            (          'Coal',           'Coal'),
            (         'Hydro',          'Hydro'),
            (         'Solar',          'Solar'),
            (          'Wind',           'Wind'),
            (         'India', 'Bheramara HVDC'),
            (         'India',        'Tripura'),
            (         'India',          'Adani'),
            (         'Nepal',          'Nepal'),
            (       'Remarks',        'Remarks')],
           )


In [9]:
data_downlaoded = pd.read_csv("powergrid_data.csv")
data_downlaoded.columns

Index(['Date', 'Time', 'Generation(MW)', 'Demand(MW)', 'Loadshed', 'Gas',
       'Liquid Fuel', 'Coal', 'Hydro', 'Solar', 'Wind', 'India', 'India.1',
       'India.2', 'Nepal', 'Remarks'],
      dtype='object')

In [10]:
data.columns

MultiIndex([(          'Date',           'Date'),
            (          'Time',           'Time'),
            ('Generation(MW)', 'Generation(MW)'),
            (    'Demand(MW)',     'Demand(MW)'),
            (      'Loadshed',       'Loadshed'),
            (           'Gas',            'Gas'),
            (   'Liquid Fuel',    'Liquid Fuel'),
            (          'Coal',           'Coal'),
            (         'Hydro',          'Hydro'),
            (         'Solar',          'Solar'),
            (          'Wind',           'Wind'),
            (         'India', 'Bheramara HVDC'),
            (         'India',        'Tripura'),
            (         'India',          'Adani'),
            (         'Nepal',          'Nepal'),
            (       'Remarks',        'Remarks')],
           )